# Quantization Benchmark

This notebook compares a baseline HF model against a 4-bit load path when the runtime supports it.
The defaults are lightweight so the notebook can still run on a laptop, but the code paths map directly to a 7B GPU setup.

In [ ]:
from pathlib import Path
import os
import sys

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / 'llm_lab').exists():
        sys.path.insert(0, str(candidate))
        repo_root = candidate
        break

from llm_lab.benchmarks import memory_snapshot, measure_generation, perplexity_from_texts, estimate_parameter_size_mb
from llm_lab.models import load_causal_lm

model_name = os.environ.get('LLM_LAB_QUANT_MODEL', 'sshleifer/tiny-gpt2')
prompt = 'Explain quantization in one concise paragraph.'
eval_texts = [
    'Quantization reduces the number of bits used to represent weights.',
    'Lower precision usually saves memory and bandwidth.',
]
print(f'Repo root: {repo_root}')
print(f'Model: {model_name}')
print('Initial memory snapshot:', memory_snapshot())

In [ ]:
baseline = load_causal_lm(model_name, quantized=False, device_map=None)
baseline_model = baseline.model
baseline_tokenizer = baseline.tokenizer
baseline_metrics = {
    'memory_mb': estimate_parameter_size_mb(baseline_model),
    'perplexity': perplexity_from_texts(baseline_model, baseline_tokenizer, eval_texts).value,
    'tokens_per_second': measure_generation(baseline_model, baseline_tokenizer, prompt).value,
}
baseline_metrics

In [ ]:
quantized_metrics = None
try:
    quantized = load_causal_lm(model_name, quantized=True)
    quantized_metrics = {
        'memory_mb': estimate_parameter_size_mb(quantized.model),
        'perplexity': perplexity_from_texts(quantized.model, quantized.tokenizer, eval_texts).value,
        'tokens_per_second': measure_generation(quantized.model, quantized.tokenizer, prompt).value,
    }
except Exception as exc:
    quantized_metrics = {'status': 'quantized path skipped', 'reason': str(exc)}
quantized_metrics

Swap `model_name` to `mistralai/Mistral-7B-v0.1` on a CUDA machine with `bitsandbytes` installed to reproduce the large-model comparison and log the full memory drop plus perplexity delta.